# Clase 167 — ONNX + ONNX Runtime: portabilidad

Entrenamos sklearn LogReg, exportamos a ONNX (con fallback) y benchmarkeamos inferencia.

In [ ]:
import numpy as np, time, json, pickle
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
rng = np.random.default_rng(42)

X, y = make_classification(n_samples=2000, n_features=20, n_informative=10, random_state=42)
X_train, X_test = X[:1500], X[1500:]
y_train, y_test = y[:1500], y[1500:]

clf = LogisticRegression(max_iter=500).fit(X_train, y_train)
print(f'sklearn acc test: {clf.score(X_test, y_test):.3f}')

## 1. Exportar a ONNX (con fallback a pickle + JSON manifest)

In [ ]:
ONNX_OK = False; ORT_OK = False
try:
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import FloatTensorType
    initial = [('input', FloatTensorType([None, X_train.shape[1]]))]
    onnx_model = convert_sklearn(clf, initial_types=initial)
    with open('model.onnx', 'wb') as f: f.write(onnx_model.SerializeToString())
    ONNX_OK = True; print(f'export ONNX OK ({len(onnx_model.SerializeToString())} bytes)')
except Exception as e:
    print(f'skl2onnx no disponible: {type(e).__name__}: {str(e)[:80]}')
    print('→ fallback: pickle + JSON manifest del schema')

In [ ]:
# Fallback siempre disponible: pickle + manifest
manifest = {
    'input': {'name': 'input', 'shape': [None, int(X_train.shape[1])], 'dtype': 'float32'},
    'output': {'name': 'label', 'shape': [None], 'dtype': 'int64'},
    'framework': 'sklearn', 'sklearn_class': 'LogisticRegression',
}
with open('model.pkl', 'wb') as f: pickle.dump(clf, f)
with open('manifest.json', 'w') as f: json.dump(manifest, f, indent=2)
print(json.dumps(manifest, indent=2))

## 2. Cargar con ONNX Runtime y verificar paridad

In [ ]:
try:
    import onnxruntime as ort
    if ONNX_OK:
        sess = ort.InferenceSession('model.onnx', providers=['CPUExecutionProvider'])
        input_name = sess.get_inputs()[0].name
        pred_ort = sess.run(None, {input_name: X_test.astype(np.float32)})[0]
        pred_sk = clf.predict(X_test)
        match = np.mean(pred_ort == pred_sk)
        print(f'paridad ONNX vs sklearn: {match:.2%}')
        ORT_OK = True
    else:
        print('skip: no hay ONNX exportado')
except Exception as e:
    print(f'onnxruntime no disponible: {type(e).__name__}')
    print('→ verificación de paridad omitida; pickle round-trip:')
    clf2 = pickle.load(open('model.pkl', 'rb'))
    print(f'  pickle round-trip acc: {clf2.score(X_test, y_test):.3f}')

## 3. Benchmark: sklearn vs ONNX Runtime

In [ ]:
def bench(fn, X, n=50):
    t0 = time.perf_counter()
    for _ in range(n): fn(X)
    return (time.perf_counter() - t0) / n * 1000   # ms/run

X32 = X_test.astype(np.float32)
ms_sk = bench(lambda x: clf.predict(x), X_test)
print(f'sklearn predict:  {ms_sk:7.3f} ms/run ({len(X_test)} samples)')
if ORT_OK:
    ms_ort = bench(lambda x: sess.run(None, {input_name: x})[0], X32)
    print(f'ONNX Runtime:     {ms_ort:7.3f} ms/run')
    print(f'speedup ORT/sklearn: {ms_sk/ms_ort:.2f}x')
else:
    print('ONNX Runtime no disponible — bench omitido')

## 4. Schema introspection

In [ ]:
if ORT_OK:
    print('--- ONNX inputs ---')
    for inp in sess.get_inputs():
        print(f'  {inp.name}: shape={inp.shape}, type={inp.type}')
    print('--- ONNX outputs ---')
    for out in sess.get_outputs():
        print(f'  {out.name}: shape={out.shape}, type={out.type}')
else:
    print('Schema desde manifest.json:')
    print(json.dumps(manifest, indent=2))

## 5. Portabilidad cross-platform

Un mismo `.onnx` corre en:

| Target | Runtime |
|---|---|
| Web | onnxruntime-web (WASM/WebGL/WebGPU) |
| iOS / Android | onnxruntime-mobile, Core ML, NNAPI |
| Server CPU | onnxruntime (default) |
| Server GPU | onnxruntime-gpu (CUDA/TensorRT/DirectML) |
| Edge | ONNX Runtime + ARM, Jetson, Snapdragon |
| C++ / Rust / Go / Java | bindings nativos |

**Workflow típico:** train (PyTorch/TF/sklearn) → export ONNX → optimize (`onnxruntime.transformers.optimizer`) → deploy.

## Ejercicio guiado

1. Cuantizar el modelo ONNX a int8 con `onnxruntime.quantization.quantize_dynamic`. Comparar latencia y accuracy.
2. Exportar un RandomForest y verificar paridad.
3. Exportar un PyTorch MLP con `torch.onnx.export` y cargarlo.

## Conclusiones

- ONNX = lingua franca de modelos ML cross-framework.
- ONNX Runtime suele dar 2-10× speedup sobre el framework original en CPU.
- Para deploy multi-plataforma (web/mobile/edge) es el camino estándar.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y comentadas de los ejercicios del README. Las de **núcleo numérico** son ejecutables (con `assert` de verificación); las de frameworks/servicios no instalados aquí (TF, PyTorch, diffusers, Gymnasium, GCP…) se muestran como **código real de referencia** listo para copiar en un entorno con esas dependencias.

### Ejercicio 1 — PyTorch → ONNX

```python
import torch
torch.onnx.export(model, dummy_input, 'model.onnx', opset_version=17,
                  input_names=['input'], output_names=['output'],
                  dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}})
```

### Ejercicio 2 — TensorFlow → ONNX (CLI)

```bash
python -m tf2onnx.convert --saved-model dir/ --output model.onnx --opset 17
```

### Ejercicio 3 — Inferencia con ONNX Runtime

```python
import onnxruntime as ort, numpy as np
sess = ort.InferenceSession('model.onnx',
                            providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
out = sess.run(None, {'input': x.astype('float32')})[0]
# Verificar vs framework original: np.testing.assert_allclose(out, ref, atol=1e-4)
```

### Ejercicio 4 — Graph optimization

```python
so = ort.SessionOptions()
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
sess = ort.InferenceSession('model.onnx', so)
# Fusiona nodos (Conv+BN+ReLU), elimina nodos muertos, constant folding.
```

### Ejercicio 5 — Quantization dinámica (núcleo int8 a mano, ejecutable)

`quantize_dynamic` cuantiza los pesos a int8 (4× menos memoria). Debajo mostramos el **mecanismo real** (escala simétrica + round + clip) en NumPy y verificamos el error, y luego la llamada de ONNX Runtime.

In [ ]:
import numpy as np
rng = np.random.default_rng(3)
W = rng.standard_normal((128, 128)).astype(np.float32)   # pesos float32

def quantize_int8_symmetric(W):
    amax = np.abs(W).max()
    scale = amax / 127.0                                 # rango [-127, 127]
    q = np.clip(np.round(W / scale), -127, 127).astype(np.int8)
    return q, scale

q, scale = quantize_int8_symmetric(W)
W_deq = q.astype(np.float32) * scale                     # dequantize
err = np.abs(W - W_deq).max()

assert q.dtype == np.int8
assert W.nbytes == 4 * q.nbytes                          # float32=4 bytes, int8=1 byte
assert err < scale                                       # error acotado por ~medio paso
print('int8: %d -> %d bytes (4x menos), error max %.4f' % (W.nbytes, q.nbytes, err))
print('OK: cuantizacion int8 simetrica verificada')

En ONNX Runtime la cuantización dinámica es una línea:

```python
from onnxruntime.quantization import quantize_dynamic, QuantType
quantize_dynamic('model.onnx', 'model_q.onnx', weight_type=QuantType.QUInt8)
```